# Notebook 04 — SHAP Explainability
## AeroTwinML · Understanding Model Predictions

**Objective:** Use SHAP (SHapley Additive exPlanations) to understand:
1. Which features matter most for each horizon
2. How each feature pushes predictions up/down
3. Why Hyderabad and Karachi get different forecasts

**Why SHAP?**
- Game-theory based — fair attribution of feature importance
- Works for any model type (tree, linear, etc.)
- Shows direction (positive/negative contribution)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils.config import get
from utils.storage import load_parquet
from feature_store.feature_builder import FeatureBuilder
from models.trainer import build_models_for_horizons, find_best_models_per_horizon

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

# Load and build features
DATA_DIR = Path(get('storage.data_dir', '../data'))
df = load_parquet(DATA_DIR / 'processed' / 'merged_hourly' / 'merged_latest.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])

builder = FeatureBuilder(df)
featured = builder.build_all()
train_df = builder.get_training_data()

# Prepare features
exclude = ('timestamp', 'source', 'station_name', 'city', 'country',
           'dominant_pollutant', 'merged_at', 'fetched_at', 'latitude', 'longitude')
feature_cols = [
    c for c in featured.columns
    if not c.startswith('target_')
    and c not in exclude
    and featured[c].dtype in ('float64', 'float32', 'int64', 'int32')
]

# Train models
split_idx = int(len(train_df) * 0.8)
train_split = train_df.iloc[:split_idx]
test_split = train_df.iloc[split_idx:]

target_cols = {'24h': 'target_aqi_24h', '48h': 'target_aqi_48h', '72h': 'target_aqi_72h'}
results = build_models_for_horizons(feature_cols, target_cols, train_split, test_split)
best_per_horizon = find_best_models_per_horizon(results)

print('Models to explain:')
for h, entry in best_per_horizon.items():
    print(f'  {h}: {entry["model_name"]} (RMSE={entry["rmse"]:.3f})')

## 1. SHAP Analysis for Best 24h Model

In [ ]:
import shap

# Get the best 24h model
model_24h = best_per_horizon['24h']['model']
model_name = best_per_horizon['24h']['model_name']

# Prepare test data
X_test = test_split[feature_cols].fillna(0)

# Get the underlying sklearn model for SHAP
if hasattr(model_24h, 'model'):
    sklearn_model = model_24h.model
else:
    sklearn_model = model_24h

# Choose explainer based on model type
model_type = type(sklearn_model).__name__
print(f'Model: {model_name} ({model_type})')

if 'Forest' in model_type or 'Boost' in model_type or 'XGB' in model_type or 'LGBM' in model_type:
    explainer = shap.TreeExplainer(sklearn_model)
else:
    explainer = shap.KernelExplainer(sklearn_model.predict, X_test.sample(min(100, len(X_test))))

shap_values = explainer.shap_values(X_test)
print(f'SHAP values shape: {shap_values.shape}')

## 2. Global Feature Importance (SHAP Summary)

In [ ]:
# Summary plot
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.title(f'SHAP Summary — {model_name} (24h horizon)')
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot of mean |SHAP|
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, plot_type='bar', show=False)
plt.title(f'Mean |SHAP| — {model_name} (24h horizon)')
plt.tight_layout()
plt.show()

## 3. Per-Horizon Feature Importance Comparison

In [ ]:
# Compare top features across horizons
importance_by_horizon = {}

for horizon, entry in best_per_horizon.items():
    model = entry['model']
    underlying = model.model if hasattr(model, 'model') else model
    
    try:
        if hasattr(underlying, 'feature_importances_'):
            imp = pd.Series(underlying.feature_importances_, index=feature_cols)
            importance_by_horizon[horizon] = imp
    except Exception as e:
        print(f'Skipping {horizon}: {e}')

if importance_by_horizon:
    imp_df = pd.DataFrame(importance_by_horizon)
    top_features = imp_df.mean(axis=1).nlargest(15).index
    
    fig, ax = plt.subplots(figsize=(14, 8))
    imp_df.loc[top_features].plot(kind='barh', ax=ax)
    ax.set_title('Feature Importance by Horizon (Top 15)')
    ax.set_xlabel('Importance')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

## 4. City-Specific Predictions Explained

Why do Hyderabad and Karachi get different forecasts?

In [ ]:
if 'city' in test_split.columns:
    # Get predictions for each city
    for city in test_split['city'].unique():
        city_mask = test_split['city'] == city
        city_X = X_test[city_mask.values]
        
        if len(city_X) == 0:
            continue
        
        # Get SHAP for last row of this city
        city_shap = explainer.shap_values(city_X.tail(1))
        
        # Top drivers
        shap_vals = pd.Series(city_shap[0], index=feature_cols)
        top = shap_vals.abs().nlargest(5)
        
        print(f'\n=== {city} — Top 5 drivers for latest prediction ===')
        for feat in top.index:
            val = shap_vals[feat]
            direction = '+' if val > 0 else '-'
            print(f'  {feat:30s} SHAP={val:+.3f} ({direction})')
        print(f'  Predicted AQI: {sklearn_model.predict(city_X.tail(1))[0]:.1f}')

## 5. Waterfall Plot for Single Prediction

In [ ]:
# Waterfall for the latest test sample
latest_idx = -1
print(f'Explaining prediction for test sample {latest_idx}')

try:
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[latest_idx],
            base_values=explainer.expected_value,
            data=X_test.iloc[latest_idx].values,
            feature_names=feature_cols,
        ),
        max_display=15,
        show=False,
    )
    plt.title('SHAP Waterfall — Latest Prediction')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Waterfall not available: {e}')
    # Fallback: force plot
    shap.force_plot(explainer.expected_value, shap_values[latest_idx], X_test.iloc[latest_idx], feature_names=feature_cols, matplotlib=True)
    plt.show()

## Summary

**Key insights from SHAP:**
- Weather variables (temperature, humidity, wind) are top drivers
- Lag features (AQI at t-24h) provide strong predictive signal
- City encoding helps differentiate between Hyderabad and Karachi
- Different horizons rely on different features

**Next:** Notebook 05 — Inference and Forecasting